# OptiCrop: Smart Agricultural Production Optimization Engine
### AI/ML Track | GPCET | Team Lead: P Hemanth
**Dataset:** https://www.kaggle.com/datasets/chitrakumari25/smart-agricultural-production-optimizing-engine

---
## Epic 2: Data Collection and Analysis
---

### Task 1: Download the Dataset
- **Source:** Kaggle — Smart Agricultural Production Optimizing Engine
- **Link:** https://www.kaggle.com/datasets/chitrakumari25/smart-agricultural-production-optimizing-engine
- **File:** Crop_recommendation.csv | **Records:** 2200 | **Features:** N, P, K, temperature, humidity, pH, rainfall, label

### Task 2: Importing the Libraries
Required libraries for data analysis, ML operations, and visualization are imported below.

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('max_colwidth', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('fivethirtyeight')
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from ipywidgets import interact
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

: 

### Task 3: Read the Dataset
The dataset is loaded using `read_csv()` and explored using `head()`, `info()`, and `describe()`.

In [ ]:
data = pd.read_csv('Crop_recommendation.csv')
df = data.copy()
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
print('Unique Crops:', df['label'].nunique())
print('Crop List:', sorted(df['label'].unique()))

In [ ]:
df.describe()

### Task 4: Univariate Analysis
Distplot and countplot are used to analyze individual feature distributions.

In [ ]:
features = ['N','P','K','temperature','humidity','ph','rainfall']
colors_list = ['gold','slateblue','lightpink','lightgreen','royalblue','tomato','yellow']
labels = ['Ratio of Nitrogen','Ratio of Phosphorous','Ratio of Potassium','Ratio of Temperature','Ratio of Humidity','Ratio of PH','Ratio of Rainfall']
fig, axes = plt.subplots(2, 4, figsize=(18,10))
fig.suptitle('Distribution of Agricultural Conditions', fontsize=16, fontweight='bold')
axes = axes.flatten()
for i,(feature,color,label) in enumerate(zip(features,colors_list,labels)):
    axes[i].hist(df[feature], bins=20, color=color, alpha=0.7, edgecolor='white')
    axes[i].set_xlabel(label, fontsize=9)
    axes[i].set_ylabel('Density', fontsize=9)
axes[7].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14,6))
sns.countplot(x='label', data=df, palette='Set2')
plt.title('Count of Each Crop in Dataset', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Task 5: Bivariate Analysis
Scatter plots are used to identify relationships between two agricultural features.

In [ ]:
plt.figure(figsize=(12,10))
sns.scatterplot(x=df['humidity'], y=df['label'], alpha=0.6)
plt.title('Bivariate Analysis: Humidity vs Crop Label', fontsize=14, fontweight='bold')
plt.xlabel('humidity')
plt.ylabel('label')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,10))
sns.scatterplot(x=df['temperature'], y=df['label'], color='tomato', alpha=0.6)
plt.title('Bivariate Analysis: Temperature vs Crop Label', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Task 6: Multivariate Analysis
Countplot and correlation heatmap are used to study multiple feature relationships simultaneously.

In [ ]:
sns.countplot(df)
plt.title('Multivariate Analysis: Feature Count', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df[features].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Epic 3: Data Pre-Processing
---

### Task 1: Checking for Null Values
Using `df.shape`, `df.info()`, and `df.isnull().sum()` to check dataset structure and missing values.

In [ ]:
df.isnull().sum()

In [ ]:
print('No null values found! Dataset is clean and ready for preprocessing.')

### Task 2: Handling Outliers
Boxplot is used to visualize outliers. IQR method is applied to treat outliers in the phosphorous feature.

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(df)
plt.title('Boxplot - Outlier Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# IQR method to handle outliers in phosphorous
Q1 = df['P'].quantile(0.25)
Q3 = df['P'].quantile(0.75)
IQR = Q3 - Q1   # IQR is interquartile range
filter = (df['P'] >= Q1 - 1.5 * IQR) & (df['P'] <= Q3 + 1.5 * IQR)
df = df.loc[filter]
print('Shape after outlier removal:', df.shape)

### Task 3: Extracting Seasonal Crops
Crops are grouped based on seasonal environmental conditions — Summer, Winter, and Rainy.

In [ ]:
print('Summer crops')
print(df[(df['temperature']>30) & (df['humidity']>50)]['label'].unique())
print('--------------------------------------------------')
print('Winter crops')
print(df[(df['temperature']<20) & (df['humidity']>30)]['label'].unique())
print('--------------------------------------------------')
print('Rainy crops')
print(df[(df['rainfall']>200) & (df['humidity']>50)]['label'].unique())
print('--------------------------------------------------')

### Task 4: Splitting Data into Train and Test Sets
The dataset is split into feature variables (X) and target variable (y), then divided into 80% train and 20% test.

In [ ]:
y = df['label']
x = df.drop(['label'], axis=1)
print('Shape of x', x.shape)
print('Shape of x=y', y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
print('The shape of x train', x_train.shape)
print('The shape of x test', x_test.shape)
print('The shape of y train', x_train.shape)
print('The shape of y test', x_test.shape)

---
## Epic 4: Model Building
---

### Task 1: K-Means Clustering
Elbow method is used to find the optimal number of clusters. K-Means is then applied to group similar agricultural conditions.

In [20]:
# Elbow method to find optimal number of clusters
plt.rcParams['figure.figsize'] = (10, 4)
wcss = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=0)
    km.fit(x)
    wcss.append(km.inertia_)
plt.plot(range(1, 11), wcss)
plt.title('The Elbow method', fontsize=20)
plt.xlabel('No of clusters')
plt.ylabel('WCSS')
plt.show()

In [21]:
# Train K-Means with optimal clusters = 4
km = KMeans(n_clusters=4, init='k-means++', max_iter=300, n_init=10, random_state=0)
y_means = km.fit_predict(x)
a = df['label']
y_means = pd.DataFrame(y_means)
z = pd.concat([y_means, a], axis=1)
z = z.rename(columns={0: 'cluster'})
print('lets check the results after applying the K-Means clustering analysis \n')
print('Crops in First cluster:', z[z['cluster']==0]['label'].unique())
print('------------------------------------------------------')
print('Crops in Second cluster:', z[z['cluster']==1]['label'].unique())
print('------------------------------------------------------')
print('Crops in Third cluster:', z[z['cluster']==2]['label'].unique())
print('------------------------------------------------------')
print('Crops in Fourth cluster:', z[z['cluster']==3]['label'].unique())

### Task 2: Logistic Regression
Logistic Regression is trained using `fit()` and predictions are generated using `predict()`.

In [22]:
# LOGISTIC REGRESSION
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
print('Logistic Regression model trained successfully!')

### Task 3: Evaluating Model Performance and Saving the Best Model
Model is evaluated using classification metrics. Best model is saved using Pickle.

In [23]:
from sklearn.metrics import classification_report
cr = classification_report(y_test, y_pred)
print(cr)

In [24]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print('Model Accuracy:', model.score(x_test, y_test))

In [25]:
# Save the best model using Pickle
pickle.dump(model, open('model.pkl', 'wb'))
print('Model saved as model.pkl successfully!')

### Task 4: Predict the Best Crop Based on Given Parameters
The saved model is used to predict the most suitable crop based on user-provided environmental parameters.

In [26]:
# Load saved model and predict
model = pickle.load(open('model.pkl', 'rb'))
prediction = model.predict((np.array([[105, 35, 40, 25, 64, 7, 160]])))
print('The suggested crop for given climatic condition is:', prediction)

---
## Epic 5: Application Building
---

### Task 1: Building HTML Pages
Three HTML pages are created:
- **home.html** — Navigation with Home, About, FindYourCrop links
- **about.html** — Project description and objectives
- **findyourcrop.html** — Crop prediction input form with N, P, K, temperature, humidity, pH, rainfall fields

All pages use Flask's `url_for()` for routing and `{{ prediction_text }}` to display results.

### Task 2: Build Python Backend Code
Flask backend (`app.py`) is created with routes for Home, About, FindYourCrop, and Predict.

```python
import os
import numpy as np
from flask import Flask, request, render_template
import pickle

app = Flask(__name__)
model = pickle.load(open('model.pkl', 'rb'))

@app.route('/')
def home():
    return render_template('home.html')

@app.route('/about')
def about():
    return render_template('about.html')

@app.route('/findyourcrop')
def findyourcrop():
    return render_template('findyourcrop.html')

@app.route('/predict', methods=['POST'])
def predict():
    int_features = [float(x) for x in request.form.values()]
    features = [np.array(int_features)]
    prediction = model.predict(features)
    output = prediction[0]
    return render_template('findyourcrop.html',
        prediction_text='Best crop for given conditions is {}'.format(output))

if __name__ == "__main__":
    port = int(os.environ.get('PORT', 5000))
    app.run(host='0.0.0.0', port=port, debug=False)
```

### Task 3: Run the Application
**Steps to run the OptiCrop Flask Application:**

1. Open the project folder in VS Code
2. Ensure `model.pkl`, `app.py`, and `templates/` folder with HTML files are present
3. Install dependencies:
```bash
pip install flask scikit-learn pandas numpy
```
4. Run the application:
```bash
python app.py
```
5. Click the server URL shown in terminal: `http://127.0.0.1:5000`

**Pages:**
- **Home Page** — Displays OptiCrop intro with navigation
- **About Page** — Displays project description
- **FindYourCrop Page** — Enter soil parameters → Click Predict → Get crop recommendation

**Output:** `Best crop for given conditions is coffee`